In [68]:
import pandas as pd
import matplotlib.pyplot as plt

In [69]:
import torch
import torch.nn as nn
from sklearn.ensemble import GradientBoostingClassifier

In [70]:
# Use Metal Performance Shaders if available (GPU acceleration).
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device to be used for training: {device}")

Device to be used for training: mps


In [71]:
# Read in the cleaned data
numeric_dtypes = {
    "Best of": "Int64",
    "Rank_1": "Int64",
    "Rank_2": "Int64",
    "Pts_1": "Int64",
    "Pts_2": "Int64",
    "Odd_1": "Float64",
    "Odd_2": "Float64",
    "Margin": "Float64",
    "Prob_1": "Float64",
    "Prob_2": "Float64",
    "WinRate180_1": "Float64",
    "WinRateLast5_365_1": "Float64",
    "WinRate180_2": "Float64",
}

wta = pd.read_csv(
    "../data/clean_wta.csv",
    dtype=numeric_dtypes,
    parse_dates=["Date"],
    na_values=["", "NA", "N/A", "null", "None", "-"],
    keep_default_na=True,
)


In [ ]:
# Create Chronological train(80%)/validation(10%)/test(10%) splits

# Choose date boundaries near 80% and 90% of the wta rows.
validation_start = wta["Date"].iloc[int(len(wta) * 0.8)]
test_start = wta["Date"].iloc[int(len(wta) * 0.9)]

train = wta.loc[wta["Date"] < validation_start].copy() # Train split

validation = wta.loc[(wta["Date"] >= validation_start) & (wta["Date"] < test_start)].copy() # Validation split

test = wta.loc[wta["Date"] >= test_start].copy() # Test split

# Check that all wta rows are assigned and the date ranges do not overlap.

assert len(train) + len(validation) + len(test) == len(wta)

assert train["Date"].max() < validation["Date"].min()

assert validation["Date"].max() < test["Date"].min()

split_summary = pd.DataFrame([
    {
        "Split": name,
        "Rows": len(part),
        "Percent": round(100 * len(part) / len(wta), 2),
        "Start": part["Date"].min(),
        "End": part["Date"].max(),
    }
    for name, part in [
        ("Train", train), ("Validation", validation), ("Test", test)
    ]
])

display(split_summary)

In [73]:
# Establish Two Baselines for Metric Comparisons. Ties are resolved simply by choosing Player 1.

# 1. Picking the higher ranked player.
# A smaller rank number means a higher-ranked player.

rank_preds = wta["Player_1"].where(
    wta["Rank_1"] <= wta["Rank_2"],
    wta["Player_2"],
)

# 2. the bookmaker's odds against actual match victory.

bookmaker_prob_preds = wta["Player_1"].where(
    wta["Prob_1"] >= wta["Prob_2"],
    wta["Player_2"],
)

In [86]:
# Create inputs and targets for the outcome prediction task.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import TensorDataset

basic_features = ["Court", "Surface", "Rank_1", "Rank_2"]
basic_features_enhanced = basic_features + ["Margin"]

# Switch to basic_features_enhanced to include bookmaker overround (Margins).
features = basic_features
categorical_features = ["Court", "Surface"]
numeric_features = [name for name in features if name not in categorical_features]

# 1. Impute and standardize numerical inputs; one-hot encode categories.
# Fit every preprocessing step on training data only.
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("categorical", OneHotEncoder(
        handle_unknown="ignore", sparse_output=False
    ), categorical_features),
])

X_train_array = preprocessor.fit_transform(train[features])
X_validation_array = preprocessor.transform(validation[features])
X_test_array = preprocessor.transform(test[features])
feature_names = preprocessor.get_feature_names_out()

# 2. Target: 1 = Player 1 wins, 0 = Player 2 wins.
# CrossEntropyLoss expects a one-dimensional tensor of integer class labels.
def make_targets(matches):
    valid_winners = (
        matches["Winner"].eq(matches["Player_1"])
        | matches["Winner"].eq(matches["Player_2"])
    )
    if not valid_winners.fillna(False).all():
        raise ValueError("Each Winner must match Player_1 or Player_2.")

    labels = matches["Winner"].eq(matches["Player_1"]).astype("int64")
    return torch.tensor(labels.to_numpy(), dtype=torch.long, device=device)

# 3. Convert to tensors on the selected CPU/GPU, preserving split row order.
X_train = torch.tensor(X_train_array, dtype=torch.float32, device=device)
X_validation = torch.tensor(X_validation_array, dtype=torch.float32, device=device)
X_test = torch.tensor(X_test_array, dtype=torch.float32, device=device)

y_train = make_targets(train)
y_validation = make_targets(validation)
y_test = make_targets(test)

train_data = TensorDataset(X_train, y_train)
validation_data = TensorDataset(X_validation, y_validation)
test_data = TensorDataset(X_test, y_test)

input_size = X_train.shape[1]

for name, inputs, targets in [
    ("Train", X_train, y_train),
    ("Validation", X_validation, y_validation),
    ("Test", X_test, y_test),
]:
    assert torch.isfinite(inputs).all(), f"{name} inputs contain NaN or infinity."
    assert inputs.shape[0] == targets.shape[0]
    print(f"{name}: inputs {tuple(inputs.shape)}, targets {tuple(targets.shape)}")

print("Encoded features:", list(feature_names))


Train: inputs (35667, 9), targets (35667,)
Validation: inputs (4464, 9), targets (4464,)
Test: inputs (4464, 9), targets (4464,)
Encoded features: ['numeric__Rank_1', 'numeric__Rank_2', 'categorical__Court_Indoor', 'categorical__Court_Outdoor', 'categorical__Surface_Carpet', 'categorical__Surface_Clay', 'categorical__Surface_Grass', 'categorical__Surface_Greenset', 'categorical__Surface_Hard']


In [87]:
# Model 1: Fully Connected Neural Network with 2 hidden layers with all basic features.

class FullyConnected2lNeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, hidden_size_l2, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias=True)  # First layer
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size_l2, bias=True) # Second layer
        self.fc3 = nn.Linear(hidden_size_l2, num_classes, bias=False) # Output layer

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out

In [88]:
# Train using chronological splits; use validation loss to select the epoch.
from torch.utils.data import DataLoader

def evaluate_model(model, data_loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    model_device = next(model.parameters()).device

    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs = inputs.to(model_device)
            targets = targets.to(model_device)

            logits = model(inputs)
            loss = criterion(logits, targets)

            total_loss += loss.item() * targets.size(0)
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
            total_samples += targets.size(0)

    return total_loss / total_samples, total_correct / total_samples


def training_loop(
    model, train_data, validation_data, optimizer, criterion,
    num_epochs=10, batch_size=64,
):
    if num_epochs < 1 or len(train_data) == 0 or len(validation_data) == 0:
        raise ValueError("Use at least one epoch and non-empty training/validation sets.")

    # Preserve row order. No test data is used during training or epoch selection.
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=False)
    validation_loader = DataLoader(
        validation_data, batch_size=batch_size, shuffle=False
    )
    model_device = next(model.parameters()).device
    history = []
    best_validation_loss = float("inf")
    best_state = None
    best_epoch = None

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for inputs, targets in train_loader:
            inputs = inputs.to(model_device)
            targets = targets.to(model_device)

            optimizer.zero_grad()
            logits = model(inputs)  # CrossEntropyLoss takes logits, not softmax.
            loss = criterion(logits, targets)
            if not torch.isfinite(loss):
                raise ValueError("Training loss became non-finite.")

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * targets.size(0)
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
            total_samples += targets.size(0)

        # Training metrics aggregate batches as the weights are being updated.
        train_loss = total_loss / total_samples
        train_accuracy = total_correct / total_samples
        validation_loss, validation_accuracy = evaluate_model(
            model, validation_loader, criterion
        )
        if not torch.isfinite(torch.tensor(validation_loss)):
            raise ValueError("Validation loss became non-finite.")

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "validation_loss": validation_loss,
            "validation_accuracy": validation_accuracy,
        })

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_epoch = epoch
            # Clone so later updates cannot overwrite the saved best weights.
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }

        print(
            f"Epoch {epoch:02d}/{num_epochs} | "
            f"train loss {train_loss:.4f}, accuracy {train_accuracy:.2%} | "
            f"validation loss {validation_loss:.4f}, "
            f"accuracy {validation_accuracy:.2%}"
        )

    # Restore the best epoch for inference; do not select an epoch on test results.
    model.load_state_dict(best_state)
    model.eval()
    print(f"Restored epoch {best_epoch} (validation loss {best_validation_loss:.4f}).")
    return pd.DataFrame(history)


In [89]:
X_train.shape[1]

9

In [ ]:
# Model and optimizer setup
# First round: 60, 10
# Define hyperparameters for model 1 + input features
input_size = X_train.shape[1] # Number of encoded input features.
hidden_size = 25   # First hidden layer width.
hidden_size_l2 = 8
num_classes = 2    # Number of output classes
learning_rate = 0.01
num_epochs = 20
batch_size = 10  # Define batch size

torch.manual_seed(42)  # Reproducible initialization on the same backend.
model1 = FullyConnected2lNeuralNet(input_size, hidden_size, hidden_size_l2, num_classes).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model1.parameters(), lr=learning_rate)

In [103]:
# Train model 1 and record learning curves. Re-run the setup cell for a fresh model.
history_model1 = training_loop(
    model=model1,
    train_data=train_data,
    validation_data=validation_data,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=num_epochs,
    batch_size=batch_size,
)

history_model1


Epoch 01/20 | train loss 0.6304, accuracy 64.75% | validation loss 0.6408, accuracy 63.31%
Epoch 02/20 | train loss 0.6246, accuracy 65.10% | validation loss 0.6400, accuracy 63.37%
Epoch 03/20 | train loss 0.6237, accuracy 65.14% | validation loss 0.6404, accuracy 62.54%
Epoch 04/20 | train loss 0.6229, accuracy 65.26% | validation loss 0.6412, accuracy 62.39%
Epoch 05/20 | train loss 0.6227, accuracy 65.30% | validation loss 0.6401, accuracy 62.39%
Epoch 06/20 | train loss 0.6225, accuracy 65.30% | validation loss 0.6396, accuracy 62.59%
Epoch 07/20 | train loss 0.6226, accuracy 65.23% | validation loss 0.6386, accuracy 62.86%
Epoch 08/20 | train loss 0.6222, accuracy 65.35% | validation loss 0.6385, accuracy 62.77%
Epoch 09/20 | train loss 0.6223, accuracy 65.16% | validation loss 0.6386, accuracy 62.59%
Epoch 10/20 | train loss 0.6222, accuracy 65.25% | validation loss 0.6389, accuracy 62.61%
Epoch 11/20 | train loss 0.6221, accuracy 65.32% | validation loss 0.6383, accuracy 62.75%

,epoch,train_loss,train_accuracy,validation_loss,validation_accuracy
0,1,0.630416,0.647545,0.640844,0.633065
1,2,0.624602,0.651022,0.639974,0.633737
2,3,0.623683,0.651358,0.640430,0.625448
3,4,0.622908,0.652620,0.641183,0.623880
4,5,0.622743,0.652985,0.640106,0.623880
5,6,0.622541,0.653013,0.639571,0.625896
6,7,0.622570,0.652284,0.638623,0.628584
7,8,0.622245,0.653461,0.638507,0.627688
8,9,0.622306,0.651639,0.638639,0.625896
9,10,0.622181,0.652508,0.638879,0.626120


In [106]:
# Now use model 1 on the test set to predict the outcome and calculate accuracy.
# Run after training, which restores the best validation-loss weights.
model1.eval()

with torch.no_grad():
    test_logits = model1(X_test)
    test_probabilities = torch.softmax(test_logits, dim=1)
    model1_test_preds = test_logits.argmax(dim=1)
    model1_test_correct = (model1_test_preds == y_test).sum().item()

model1_test_accuracy = model1_test_correct / len(y_test)

# Class 1 = Player 1 wins; class 0 = Player 2 wins.
# Preserve the test DataFrame's index and row order.
test_results = test[["Date", "Player_1", "Player_2", "Winner"]].copy()
test_results["Predicted_class"] = model1_test_preds.cpu().numpy()
test_results["Model1_Prob_1"] = test_probabilities[:, 1].cpu().numpy()
test_results["Predicted_winner"] = test_results["Player_1"].where(
    test_results["Predicted_class"].eq(1), test_results["Player_2"]
)
test_results["Correct"] = test_results["Predicted_winner"].eq(test_results["Winner"])

print(
    f"Model 1 test accuracy: {model1_test_accuracy:.2%} "
    f"({model1_test_correct:,}/{len(y_test):,} matches)"
)
display(test_results.head())


Model 1 test accuracy: 62.95% (2,810/4,464 matches)


,Date,Player_1,Player_2,Winner,Predicted_class,Model1_Prob_1,Predicted_winner,Correct
40131,2024-07-17,Avanesyan E.,Masarova R.,Avanesyan E.,1,0.529909,Avanesyan E.,True
40132,2024-07-17,Monnet C.,Lamens S.,Lamens S.,0,0.445265,Lamens S.,True
40133,2024-07-17,Sasnovich A.,Timofeeva M.,Sasnovich A.,0,0.466919,Timofeeva M.,False
40134,2024-07-17,Sorribes Tormo S.,Schmiedlova A.,Schmiedlova A.,0,0.500000,Schmiedlova A.,True
40135,2024-07-17,Andreeva E.,Semenistaja D.,Andreeva E.,1,0.512065,Andreeva E.,True


In [108]:
# TODO: Implement grid search over the hyperparameters. 

In [105]:
rank_accuracy = rank_preds.loc[test.index].eq(test["Winner"]).mean()
bookmaker_accuracy = bookmaker_prob_preds.loc[test.index].eq(test["Winner"]).mean()
rank_accuracy_train = rank_preds.loc[train.index].eq(train["Winner"]).mean()
bookmaker_accuracy_train = bookmaker_prob_preds.loc[train.index].eq(train["Winner"]).mean()

print(f"Rank baseline accuracy (test/train splits): {rank_accuracy:.2%}, {rank_accuracy_train:.2%}")
print(f"Bookmaker baseline accuracy (test/train splits): {bookmaker_accuracy:.2%}, {bookmaker_accuracy_train:.2%}")

Rank baseline accuracy (test/train splits): 63.13%, 65.49%
Bookmaker baseline accuracy (test/train splits): 68.62%, 69.26%


In [84]:
# find out for how many records, win_rate_last180d_1, win_rate_last180d_2, win_rate_last5g_1, win_rate_last5g_2 values don't exist, and drop the records that don't have all four values.

cols = [
    "win_rate_last180d_1",
    "win_rate_last180d_2",
    "win_rate_last5g_1",
    "win_rate_last5g_2",
]

# Missing values in each feature.
print(wta[cols].isna().sum())

# Records missing at least one of these features.
print("Missing any:", wta[cols].isna().any(axis=1).sum())

# Records missing all four features.
print("Missing all:", wta[cols].isna().all(axis=1).sum())

win_rate_last180d_1    1707
win_rate_last180d_2    1631
win_rate_last5g_1      1028
win_rate_last5g_2       999
dtype: int64
Missing any: 3105
Missing all: 106


In [ ]:
# Compute additional metrics such as accuracy, precision, recall, and f1 score.

In [ ]:
# Make histograms.

In [ ]:
# TODO: Clean up notebook for submission.